Goal: Make baseline predictions by averaging rain, crop, and surplus data over the entire basin for a single site.
Currently using site WQS0084 but can be updated to any uid.

In [1]:
import sys
sys.path.insert(0, "../") # replace with path/to/project/root
from data import get_data, get_site_ids

import pandas as pd
import numpy as np


In [5]:
uid = "WQS0084"

data = get_data(site_uid=uid)

rain = data.rain
daily_avg_precip = ( # calculate average percipitation over entire basin for each date
    rain.groupby("date", as_index=False)["precip_in_1d"]
      .mean()
)

# Take sum of crops over the entire basin (should it be mean?)
crops = data.crops.groupby("year", as_index=False).sum() 

# Calculate mean surplus nitrogen over basin in 2017. This does not work as a feature because it is constant.
nitrogen_2017 = data.surplus[data.surplus['year'] == 2017]
SURPLUS_N_2017 = nitrogen_2017['surplus_kgha'].mean()


In [6]:
crops

,year,node_id,Alfalfa,Corn,Fallow,Hay_Pasture,Nonag,Other,Small_Grains,Soybeans
0,2000,15981,0,1553215,137010,406379,94137,15309,41312,1294518
1,2001,15981,45480,1326499,129343,414007,77089,0,107676,1442068
2,2002,15981,69094,1371048,131239,684222,141366,2,8264,1136893
3,2003,15981,22390,1464432,99201,803821,54083,2945,3079,1092211
4,2004,15981,36595,1451950,34500,502726,274257,2356,21154,1218624
5,2005,15981,28985,1432081,63619,383829,423373,8979,23664,1177632
6,2006,43956,25518,718025,16047,144855,230424,4830,8154,610463
7,2007,43956,6386,755479,24,237612,277342,17534,1787,462152
8,2008,43956,42503,2650310,1629,704802,785724,20979,7125,1913745
9,2009,43956,32290,2642556,118,679138,782729,34448,10196,1945342


In [ ]:
from data.water import aggregate_by_interval

# Aggregate water levels to each calendar day. I used max so we coult track violations,
# but might need to be average for a continuous model. This could explain why linreg is so bad.
water = aggregate_by_interval(site_uid=uid, value_col="nitrate_con", interval="1D", agg_func="max").to_frame()
water["date"] = water.index.date

water = water.reset_index(drop=True)

# Create true/false column if there was a single violation in a calendar day
water["violation"] = (water.nitrate_con > 10).astype(int)
print("Violation counts:\n", water["violation"].value_counts()) # Check counts of violations
water



Violation counts:
 violation
0    2600
1     264
Name: count, dtype: int64


,nitrate_con,date,violation
0,7.92,2018-07-25,0
1,7.92,2018-07-26,0
2,7.82,2018-07-27,0
3,7.69,2018-07-28,0
4,7.47,2018-07-29,0
...,...,...,...
2859,7.46,2026-05-23,0
2860,7.25,2026-05-24,0
2861,7.02,2026-05-25,0
2862,7.03,2026-05-26,0


In [ ]:
# Make datetime stuff compatible
water["date"] = pd.to_datetime(water["date"])
daily_avg_precip["date"] = pd.to_datetime(daily_avg_precip["date"])


# Merge water data with daily average percipitation
df = water[["date", "violation", "nitrate_con"]].merge(
    daily_avg_precip,
    on="date",
    how="left"
)


In [11]:
# Add in 7,14, and 30 day rolling averages
df["rain_7d"]  = df["precip_in_1d"].rolling(7,  min_periods=1).sum()
df["rain_14d"] = df["precip_in_1d"].rolling(14, min_periods=1).sum()
df["rain_30d"] = df["precip_in_1d"].rolling(30, min_periods=1).sum()

# Crop data is yearly -> broadcast onto every day in that crop year.
df["year"] = df["date"].dt.year
df = df.merge(crops.drop(columns='node_id'), on="year", how="left")      
 
# Surplus N is a single static estimate -> constant column. Will only matter for multiple sites.
df["surplus_n_2017"] = SURPLUS_N_2017

# Use Surplus N as a feature to explain rain. This should be adjusted to be weighted across the grid, but WIP.
df["surplus_x_rain30"] = df["surplus_n_2017"] * df["rain_30d"]
 
# Autoregressive features: nitrate is autocorrelated, recent readings matter
df["nitrate_lag1"] = df["nitrate_con"]            # today's reading
df["nitrate_lag2"] = df["nitrate_con"].shift(1)
df["nitrate_lag3"] = df["nitrate_con"].shift(2)


In [ ]:
# Add in columns that allow for dummy models (yesterday's value = today's value)
df["nitrate_tomorrow"]   = df["nitrate_con"].shift(-1)
df["violation_tomorrow"] = (df["nitrate_tomorrow"] > 10).astype(int)
 
# Remove null variables
df = df.dropna(subset=["nitrate_tomorrow", "nitrate_lag3", "nitrate_con"]).reset_index(drop=True)
df  # Look at all features in dataframe

,date,violation,nitrate_con,precip_in_1d,rain_7d,rain_14d,rain_30d,year,Alfalfa,Corn,...,Other,Small_Grains,Soybeans,surplus_n_2017,surplus_x_rain30,nitrate_lag1,nitrate_lag2,nitrate_lag3,nitrate_tomorrow,violation_tomorrow
0,2018-07-27,0,7.82,0.000000,0.215556,0.215556,0.215556,2018,68622.0,2725091.0,...,9809.0,10414.0,1901375.0,80.879331,17.433989,7.82,7.92,7.92,7.69,0
1,2018-07-28,0,7.69,0.000067,0.215623,0.215623,0.215623,2018,68622.0,2725091.0,...,9809.0,10414.0,1901375.0,80.879331,17.439436,7.69,7.82,7.92,7.47,0
2,2018-07-29,0,7.47,0.051919,0.267542,0.267542,0.267542,2018,68622.0,2725091.0,...,9809.0,10414.0,1901375.0,80.879331,21.638625,7.47,7.69,7.82,7.20,0
3,2018-07-30,0,7.20,0.040875,0.308418,0.308418,0.308418,2018,68622.0,2725091.0,...,9809.0,10414.0,1901375.0,80.879331,24.944602,7.20,7.47,7.69,6.99,0
4,2018-07-31,0,6.99,0.000000,0.308418,0.308418,0.308418,2018,68622.0,2725091.0,...,9809.0,10414.0,1901375.0,80.879331,24.944602,6.99,7.20,7.47,6.60,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1495,2026-05-22,0,7.44,0.103199,0.944141,1.474646,2.516465,2026,NaN,NaN,...,NaN,NaN,NaN,80.879331,203.529978,7.44,7.97,7.85,7.46,0
1496,2026-05-23,0,7.46,0.057710,0.986566,1.532357,2.024983,2026,NaN,NaN,...,NaN,NaN,NaN,80.879331,163.779285,7.46,7.44,7.97,7.25,0
1497,2026-05-24,0,7.25,0.031582,0.617778,1.563939,2.053266,2026,NaN,NaN,...,NaN,NaN,NaN,80.879331,166.066781,7.25,7.46,7.44,7.02,0
1498,2026-05-25,0,7.02,0.096364,0.298081,1.660303,2.149630,2026,NaN,NaN,...,NaN,NaN,NaN,80.879331,173.860607,7.02,7.25,7.46,7.03,0


In [ ]:
# Check seasonality of violations. Indicate that this time series should possibly be handled as periodic/seasonal.

df["month"] = df["date"].dt.month
monthly = df.groupby("month")["violation_tomorrow"].agg(["sum", "count"])
monthly["rate"] = monthly["sum"] / monthly["count"]
print("Violations by month:\n", monthly, "\n")
 
top_month_share = monthly["sum"].max() / monthly["sum"].sum()
print(f"Largest single month holds {top_month_share:.0%} of all violations.")
if top_month_share > 0.35:
    print("WARNING: violations are heavily seasonal. A plain chronological "
          "split may starve some folds of positives -- consider checking "
          "fold-level violation counts below, or stratifying by season "
          "instead of pure date order.\n")
else:
    print("Violations are reasonably spread across the year.\n")


Violations by month:
        sum  count      rate
month                      
3        2     24  0.083333
4       26     95  0.273684
5       59    188  0.313830
6       85    176  0.482955
7       45    190  0.236842
8        8    165  0.048485
9        3    206  0.014563
10       1    217  0.004608
11       2    193  0.010363
12       3     46  0.065217 

Largest single month holds 36% of all violations.



In [16]:
cutoff = df["date"].quantile(0.8)          # last ~20% of days held out
train = df[df["date"] <= cutoff]
hold_out  = df[df["date"] >  cutoff]
 
crop_cols = [c for c in crops.columns if c not in ["year", "node_id"]]
feature_cols = (["rain_7d", "rain_14d", "rain_30d", "surplus_x_rain30",
                  "nitrate_lag1", "nitrate_lag2", "nitrate_lag3"]
                 + crop_cols)
 
X = train[feature_cols].fillna(0).values    # train on all features for training data. Fix null values
y_class = train["violation_tomorrow"].values
y_reg   = train["nitrate_tomorrow"].values
today_value = train["nitrate_con"].values   # needed for the random-walk baseline

 

The remainder is testing different baseline models on the formatted data.

In [17]:
from scipy.stats import norm
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import average_precision_score, mean_absolute_error, mean_squared_error
 

# ------------------------------------------------------------------
# 6. TIME SERIES CROSS-VALIDATION
# TimeSeriesSplit always trains on the past and tests on a later, contiguous
# block -- never shuffles, never tests on data that precedes training data.
# ------------------------------------------------------------------
N_SPLITS = 5
tscv = TimeSeriesSplit(n_splits=N_SPLITS)
 
results = {k: [] for k in [
    "dummy_most_frequent_AP", "dummy_stratified_AP",
    "gaussian_rw_classifier_AP", "logreg_AP",
    "persistence_regression_MAE", "persistence_regression_RMSE",
    "linreg_MAE", "linreg_RMSE",
]}
 
for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
    X_train, X_test = X[train_idx], X[test_idx]
    yc_train, yc_test = y_class[train_idx], y_class[test_idx]
    yr_train, yr_test = y_reg[train_idx], y_reg[test_idx]
    today_train, today_test = today_value[train_idx], today_value[test_idx]
 
    print(f"Fold {fold}: train={len(train_idx)}  test={len(test_idx)}  "
          f"test_violations={yc_test.sum()}")
    if yc_test.sum() == 0:
        print("  WARNING: this fold has zero violations -- AP undefined/misleading for it.")
 
    # --- Dummy classifiers (the honest floor) ---
    for strategy, key in [("most_frequent", "dummy_most_frequent_AP"),
                           ("stratified", "dummy_stratified_AP")]:
        dummy = DummyClassifier(strategy=strategy, random_state=0)
        dummy.fit(X_train, yc_train)
        ap = average_precision_score(yc_test, dummy.predict_proba(X_test)[:, 1])
        results[key].append(ap)
 
    # --- Gaussian random walk ---
    # sigma estimated from TRAINING differences only -- no future leakage
    train_diffs = np.diff(today_train)
    sigma = max(np.std(train_diffs), 1e-6) if len(train_diffs) > 1 else 1.0

    # classification view: P(tomorrow > 10 | today) under N(today, sigma^2)
    rw_proba = 1 - norm.cdf((10 - today_test) / sigma)
    results["gaussian_rw_classifier_AP"].append(average_precision_score(yc_test, rw_proba))
 
    # regression view: point forecast = today's value (persistence)
    persistence_pred = today_test
    results["persistence_regression_MAE"].append(mean_absolute_error(yr_test, persistence_pred))
    results["persistence_regression_RMSE"].append(np.sqrt(mean_squared_error(yr_test, persistence_pred)))
 
    # --- Logistic regression (feature-based classification) ---
    logreg = make_pipeline(StandardScaler(),
                            LogisticRegression(class_weight="balanced", max_iter=1000))
    logreg.fit(X_train, yc_train)
    ap_lr = average_precision_score(yc_test, logreg.predict_proba(X_test)[:, 1])
    results["logreg_AP"].append(ap_lr)
 
    # --- Linear regression (feature-based regression) ---
    linreg = make_pipeline(StandardScaler(), LinearRegression())
    linreg.fit(X_train, yr_train)
    lr_pred = linreg.predict(X_test)
    results["linreg_MAE"].append(mean_absolute_error(yr_test, lr_pred))
    results["linreg_RMSE"].append(np.sqrt(mean_squared_error(yr_test, lr_pred)))
 


Fold 0: train=200  test=200  test_violations=38
Fold 1: train=400  test=200  test_violations=0
Fold 2: train=600  test=200  test_violations=5
Fold 3: train=800  test=200  test_violations=56
Fold 4: train=1000  test=200  test_violations=91


/opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:1192: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:1192: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:1192: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:1192: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


In [18]:
# ------------------------------------------------------------------
# 7. SUMMARY ACROSS FOLDS
# ------------------------------------------------------------------
print("\n=== Cross-validated results (mean +/- std across folds) ===")
print("\n-- Classification (Average Precision / PR-AUC, higher is better) --")
for key in ["dummy_most_frequent_AP", "dummy_stratified_AP",
            "gaussian_rw_classifier_AP", "logreg_AP"]:
    vals = np.array(results[key])
    print(f"{key:30s}: {vals.mean():.4f} +/- {vals.std():.4f}")
 
print("\n-- Regression (MAE / RMSE in mg/L, lower is better) --")
for key in ["persistence_regression_MAE", "persistence_regression_RMSE",
            "linreg_MAE", "linreg_RMSE"]:
    vals = np.array(results[key])
    print(f"{key:30s}: {vals.mean():.4f} +/- {vals.std():.4f}")
 
# ------------------------------------------------------------------
# HOW TO READ THIS:
# - The Gaussian random walk should beat the dummy classifiers by a wide
#   margin (it has the nontrivial advantage of knowing today's value).
#   If logreg can't beat the random walk, your engineered features (rain,
#   crops, surplus N) aren't adding information beyond "what happened
#   yesterday" -- which is a meaningful, honest finding on its own.
# - Same logic for regression: linreg should beat plain persistence, or
#   your features aren't earning their keep yet.
# ------------------------------------------------------------------
 



=== Cross-validated results (mean +/- std across folds) ===

-- Classification (Average Precision / PR-AUC, higher is better) --
dummy_most_frequent_AP        : 0.1900 +/- 0.1683
dummy_stratified_AP           : 0.1902 +/- 0.1682
gaussian_rw_classifier_AP     : 0.5782 +/- 0.3708
logreg_AP                     : 0.5475 +/- 0.3809

-- Regression (MAE / RMSE in mg/L, lower is better) --
persistence_regression_MAE    : 0.6609 +/- 0.2807
persistence_regression_RMSE   : 1.7320 +/- 0.9556
linreg_MAE                    : 1.2654 +/- 0.8982
linreg_RMSE                   : 2.1876 +/- 1.0194


In [ ]:
# Ignore this cell, want to try and model on my own, but obviously haven't gotten very far.

## Import TimeSeriesSplit
from sklearn.model_selection import TimeSeriesSplit

kfold = TimeSeriesSplit(n_splits = 5)

# for train_index, test_index in kfold.split(train):
#     print("TRAIN INDEX SIZE:", train_index.shape[0])
#     print("TEST INDEX SIZE:", test_index.shape[0])
#     print()